In [6]:
import pandas as pd
import numpy as np

df_tpm = pd.read_csv("../../rna-seq/star_rsem/rsem.merged.gene_tpm.tsv", sep="\t")
df_tpm.set_index("gene_id", inplace=True)
df_tpm

,transcript_id(s),TGEV_REP1,TGEV_REP2,TGEV_REP3,WT_REP1,WT_REP2,WT_REP3
gene_id,,,,,,,
ENSSSCG00000000002,ENSSSCT00000000003,4.81,3.13,4.73,3.79,5.62,4.07
ENSSSCG00000000003,"ENSSSCT00000000004,ENSSSCT00000052628,ENSSSCT0...",2.19,2.65,3.76,1.81,2.85,4.39
ENSSSCG00000000005,ENSSSCT00000000006,1.60,2.28,1.48,1.03,1.08,0.86
ENSSSCG00000000006,ENSSSCT00000000007,0.91,1.72,1.25,0.88,1.23,1.33
ENSSSCG00000000007,"ENSSSCT00000000008,ENSSSCT00000057594,ENSSSCT0...",8.18,9.08,8.88,5.73,7.24,6.23
...,...,...,...,...,...,...,...
ENSSSCG00000051794,"ENSSSCT00000029953,ENSSSCT00000037685,ENSSSCT0...",15.97,14.91,13.70,9.19,7.75,10.58
ENSSSCG00000051795,ENSSSCT00000075332,0.04,0.03,0.06,1.03,0.88,1.46
ENSSSCG00000051796,ENSSSCT00000083535,0.00,0.00,0.13,0.00,0.00,0.00


In [7]:
df_geneWTA = pd.read_csv("WT_100kb_Agene_id")
df_geneWTA.columns = ["gene_id"]
df_geneWTA.set_index("gene_id", inplace=True)
df_tpmWTA = df_tpm.loc[df_geneWTA.index].drop(["transcript_id(s)", "TGEV_REP1", "TGEV_REP2", "TGEV_REP3"], axis=1)
df_tpmWT_A = df_tpmWTA[["WT_REP1","WT_REP2", "WT_REP1"]].mean(axis=1)

In [8]:
df_geneWTB = pd.read_csv("WT_100kb_Bgene_id")
df_geneWTB.columns = ["gene_id"]
df_geneWTB.set_index("gene_id", inplace=True)
df_tpmWTB = df_tpm.loc[df_geneWTB.index].drop(["transcript_id(s)", "TGEV_REP1", "TGEV_REP2", "TGEV_REP3"], axis=1)
df_tpmWT_B = df_tpmWTB[["WT_REP1","WT_REP2", "WT_REP1"]].mean(axis=1)
df_tpmWT_B

gene_id
ENSSSCG00000000295    2.890000
ENSSSCG00000000297    0.123333
ENSSSCG00000000298    0.860000
ENSSSCG00000000299    0.000000
ENSSSCG00000000312    0.000000
                        ...   
ENSSSCG00000051791    0.056667
ENSSSCG00000051793    0.090000
ENSSSCG00000051796    0.000000
ENSSSCG00000051797    0.013333
ENSSSCG00000051798    0.000000
Length: 10601, dtype: float64

In [9]:
df = pd.concat([df_tpmWT_A, df_tpmWT_B], axis=1)
df.columns = ["WT_A", "WT_B"]
df_p = df.melt()
df_p.columns = ["Type", "Tpm"]
df_p['Tpm'] = df_p['Tpm'].apply(lambda x: np.log(x + 0.00001))
df_p

,Type,Tpm
0,WT_A,0.768568
1,WT_A,0.045620
2,WT_A,-0.003329
3,WT_A,1.829913
4,WT_A,-0.483410
...,...,...
60615,WT_B,-2.870393
60616,WT_B,-2.407835
60617,WT_B,-11.512925
60618,WT_B,-4.316738


In [15]:
from scipy.stats import ranksums
z, p = ranksums(df['WT_A'].dropna(), df['WT_B'].dropna())
print(z, p)

58.20713279178716 0.0


In [18]:
import plotly.express as px
import plotly.io as pio
import numpy as np

fig = px.box(df_p.dropna(), 
             x="Type",
             y="Tpm", 
             color="Type"
            )

fig.update_layout(
    width=500,
    height=500,
    paper_bgcolor='white',
    plot_bgcolor='white'
)

# 添加标题，并设置标题的位置
fig.update_layout(
    title={
        'text': "A Vs B",
        'y': 0.9,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    }
)


fig.update_yaxes(range=[-15, 15])

# 将图像保存为PDF文件
pio.write_image(fig, 'WTboxplot.pdf')
fig.show()